# Grilles alternées pour les ondes en eau peu profonde

Le but de ce TD est de calculer la dynamique d'ondes d'inertie gravité en eau peu profonde, et de montrer l'intrêt des grilles décalées (staggered grid) popularisées en dynamique des fluides géophysiques par Arakawa dans les années 70.

> Arakawa, A.; Lamb, V.R. (1977). "Computational design of the basic dynamical processes of the UCLA general circulation model". Methods in Computational Physics: Advances in Research and Applications

Commençons par importer les modules nécessaires : 


In [1]:
import numpy as np
import matplotlib.pyplot as plt


### Cas 1D sans rotation



On considère les équations des ondes en eau peu profonde (shallow water)
linéarisées autour d’un état de repos, sans rotation et en une dimension d’espace, sur un domaine périodique de longueur $L$ (assimilable à un cercle de latitude) :



\begin{align}
  \partial_t u + g\, \partial_x \eta & = 0  \\
  \partial_t \eta + H\, \partial_x u & = 0,
\end{align}

  - $u(x,t)$ est la vitesse horizontale,
  - $\eta(x,t)$ est l’élévation de la surface libre,
  - $H$ est la profondeur moyenne (constante),
  - $g$ est l’accélération de la pesanteur.

In [2]:
G = 9.81 # m/s^2
H = 4 # km, echelle verticale
L = 100 # km echelle horizontale
T = 1 # temps typique pour une onde de taille L

**Question 1** : Montrer que le système admet des solutions sous forme d’ondes planes. Déterminer et interpreter la relation de dispersion. Le système est-il dispersif ?

**Question 2 :** On considère une grille de points de colocation
$x_j = j \Delta x$, $j=0,\dots,N-1$, avec $\Delta x = L/N$. Quelle est le plus petit nombre d’onde pouvant être représentée sur cette grille ? Quelle est le plus grand ?

**Question 3 :** On note $(u_j^n,\eta_j^n)$ les valeurs numériques aux points $x_j$ et au temps $t^n = n \Delta t$.  Écrire analytiquement et à l'aide d'une fonction python un schéma leapfrog centré d’ordre 2 en espace et en temps pour cette grille.

In [3]:
dt, dx = 0.01, 0.1
number_of_time_steps = int(20 * T/dt) # on fait tourner la simulation pendant 10 périodes d'oscillation 
number_of_space_steps = int(L/dx)

# Exemple des arrays pour stocker les résultats
u = np.zeros((number_of_time_steps, number_of_space_steps))
eta = np.zeros((number_of_time_steps, number_of_space_steps))

In [ ]:
def leapfrog_step(u, eta, dt, dx, step):

    """ Intégration temporelle en utilisant le schéma de Leapfrog pour les équations de Shallow Water.
    Parameters:
    u : 2D array
        Zonal velocity component for all time steps. (shape: (nt, nx))
    eta : 2D array
        surface elevation component for all time steps . (shape: (nt, nx))
    dt : float
        Time step size.
    dx : float
        Grid spacing in the x-direction.
    step : int
        Current time step index.
    """
    if step < 1:
        raise ValueError("Leapfrog scheme requires at least two previous time steps.")
    
    # Compléter ...
        

**Question 4 :** Expliquez quel problème apparaît lors de l'initialisation de la simulation et proposez une solution simple pour le résoudre. Quel est le problème avec ce shéma d'intégration.

**Question 5 :** Pour pallier à ce problème d'initialisation nous proposons les fonctions d'initialisations suivantes. Quelle est la différence entre les deux, pourquoi l'une est à privilégier ? Un calcul complet de l'erreur commise est attendu pour les deux méthodes

In [5]:
def euler_init(u, eta, dt, dx):

    """Initialize the velocity fields using a Runge-Kutta method for the shallow water equations.
    Parameters:
    u : 2D array
        Zonal velocity component for all time steps. (shape: (nt, nx))
    eta : 2D array
        Meridional velocity component for all time steps . (shape: (nt, nx))
    dt : float
        Time step size.
    dx : float
        Grid spacing in the x-direction.
    """
    # Compute the derivatives
    du_dx = (np.roll(u[0], -1) - np.roll(u[0], 1)) / ( 2 * dx) # Periodic boundary conditions
    deta_dx = (np.roll(eta[0], -1) - np.roll(eta[0], 1)) / ( 2 * dx) # Periodic boundary conditions

    # Update the velocity fields using a simple forward Euler step for initialization
    u[1] = u[0] - G * dt * deta_dx
    eta[1] = eta[0] - H * dt * du_dx

In [ ]:
def Runge_Kutta_init(u, eta, dt, dx):

    """Initialize the velocity fields using a Runge-Kutta method (mid point Euler) for the shallow water equations.
    Parameters:
    u : 2D array
        Zonal velocity component for all time steps. (shape: (nt, nx))
    eta : 2D array
        interface elevation for all time steps . (shape: (nt, nx))
    dt : float
        Time step size.
    dx : float
        Grid spacing in the x-direction.
    """
    # Compute the derivatives at the initial time step
    du_dx = (np.roll(u[0], -1) - np.roll(u[0], 1)) / ( 2 * dx) # Periodic boundary conditions
    deta_dx = (np.roll(eta[0], -1) - np.roll(eta[0], 1)) / ( 2 * dx) # Periodic boundary conditions

    # Compute the intermediate values for the Runge-Kutta method
    u_half = u[0] - 0.5 * G * dt * deta_dx
    eta_half = eta[0] - 0.5 * H * dt * du_dx

    # Compute the derivatives at the intermediate time step
    du_dx_half = (np.roll(u_half, -1) - np.roll(u_half, 1)) / (2 * dx) # Periodic boundary conditions
    deta_dx_half = (np.roll(eta_half, -1) - np.roll(eta_half, 1)) / ( 2 * dx) # Periodic boundary conditions

    # Update the velocity fields using the Runge-Kutta method
    u[1] = u[0] - G * dt * deta_dx_half
    eta[1] = eta[0] - H * dt * du_dx_half

**Question 6 :** On donne la fonction permettant de calculer les conditions initiales correspondant à un bump gaussien centré: 

\begin{equation}
\eta(t=0,x) = u(t=0,x) = \exp(-(x - \tfrac{L}{2})^2 / (2\sigma^2)) \quad \text{with} \quad \sigma = \tfrac{L}{10}
\end{equation}
Nous donnons également la solution analytique pour cette CI. Intégrer la solution pour le pas de temps fixé, un exemple de la syntaxe d'utilisation est donné.

In [ ]:
def init_cond(u,eta):
    """Conditions initiales : bump gaussien au milieu du domaine
    Parameters:
    u : 2D array
        Composante zonale de la vitesse pour tous les pas de temps. (shape: (nt, nx))
    eta : 2D array
        élévation d'interface  pour tous les pas de temps. (shape: (nt, nx))
    """
    # Créer une perturbation gaussienne au milieu du domaine
    x = np.linspace(0, L, number_of_space_steps)
    gaussian_perturbation = np.exp(-((x - L/2)**2) / (2 * (L/10)**2))

    u[0] = gaussian_perturbation
    eta[0] = gaussian_perturbation

def analytic_solution(x, t):
    
    c = np.sqrt(G * H) 
    sigma = L / 10.0
    x0 = L / 2.0
    xm = (x - c*t) % L
    xp = (x + c*t) % L
    f_m = np.exp(-((xm - x0)**2) / (2.0 * sigma**2))   # f(x-ct)
    f_p = np.exp(-((xp - x0)**2) / (2.0 * sigma**2))   # f(x+ct)
    a = 1.0 + G / c
    b = 1.0 - G / c
    u_analytic   = 0.5 * (a * f_m + b * f_p)
    eta_analytic = (c / (2.0 * G)) * (a * f_m - b * f_p)

    return u_analytic, eta_analytic

x = np.linspace(0, L, number_of_space_steps)
t = np.linspace(0, 20 * T, number_of_time_steps)
x,t = np.meshgrid(x, t)
u_analytic, eta_analytic = analytic_solution(x, t)

In [ ]:
def solve_shallow_water(dt, dx, initialization_method='Runge-Kutta'):
     
    number_of_space_steps, number_of_time_steps = int(L/dx), int(20 * T/dt)
    u = np.zeros((number_of_time_steps, number_of_space_steps))
    eta = np.zeros((number_of_time_steps, number_of_space_steps))
    init_cond(u,eta)

    init_funcs = {'Runge-Kutta': Runge_Kutta_init, 'Euler': euler_init}
    init_funcs[initialization_method](u, eta, dt, dx)
    
    # compléter ...

    return u, eta

In [9]:
u, eta = solve_shallow_water(dt, dx, initialization_method='Runge-Kutta')

In [ ]:
# Comparaison des solutions numériques et analytiques

fig, ax = plt.subplots(2,2, figsize=(8, 8))
for i, (field, analytic_field) in enumerate(zip([u, eta], [u_analytic, eta_analytic])):
    ax[i,0].imshow(field, aspect='auto', origin='lower', extent=[0, L, 0, 10*T])
    ax[i,0].set_title(f"Numerical {['u', 'eta'][i]}")
    ax[i,1].imshow(analytic_field, aspect='auto', origin='lower', extent=[0, L, 0, 10*T])
    ax[i,1].set_title(f"Analytic {['u', 'eta'][i]}")
plt.tight_layout()

**Question 7 :** On propose d'effectuer une analyse de Von Neumann en supposant des solutions de la forme
$$ u_j^n = \hat u\, e^{i(k j \Delta x - \omega n \Delta t)}, \quad \eta_j^n = \hat \eta\, e^{i(k j \Delta x - \omega n \Delta t)}.$$
- (a) Pourquoi peut-on supposer $k$ réel ? $\omega$ est-il nécessairement réel ?
- (b)  Montrer que la relation de dispersion discrète s’écrit
$$ \sin(\omega \Delta t) = \pm \frac{c \Delta t}{\Delta x} \, \sin(k \Delta x),$$
où $c = \sqrt{gH}$ est la célérité.

**Question 8:** Supposons pour le moment $w \in \mathbb{R}$ . Sous quelle condition sur le pas de temps cette hypothèse reste-elle valide ? En déduire une condition nécessaire de stabilité du schéma (condition CFL). Quel est la forme du mode dominant lorsque la solution explose . Illustrer ce phénomène à l'aide d'un choix judicieusement mal choisi des paramètres de discrétisation $\Delta t $ et $\Delta x$ : 

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 3))
for i in range(0, number_of_time_steps //10, 50):
    ax.plot(x, u[i], label='Numerical u', lw = .7)
ax.set_xlabel('x')
ax.set_ylabel('$u$')
ax.set_title('Unstable wave propagation')
inset = ax.inset_axes([0.1, 0.5, 0.2, 0.4])
inset.plot(x[:20], u[number_of_time_steps //10- 50][:20], label='Initial condition', lw = .7)
ax.indicate_inset_zoom(inset, edgecolor="black")



**Question 9 :** On considère maintenant une grille décalée ("type C") :
        $\eta$ est définie aux centres de maille $x_j$ et $u$ aux interfaces $x_{j+1/2}$.
        On note $u_{j+1/2}^n$ et $\eta_j^n$ les valeurs des champs sur cette grille.
        Écrire le schéma leapfrog centré correspondant et la fonction python associée.

```text
      xj+1/2  xj xj+1/2 
    |    |    |    |    
----o----o----o----o----o---->
    |    |    |    |    |
    η    u     η    u    η
```


In [14]:
# Exemple des arrays pour stocker les résultats
u = np.zeros((number_of_time_steps, number_of_space_steps)) # u[i,j] = u[i]_{j+1/2}
eta = np.zeros((number_of_time_steps, number_of_space_steps)) # eta[i,j] = eta[i]_{j}

In [ ]:
def leapfrog_step_staggered(u, eta, dt, dx, step):

    """ Intégration temporelle en utilisant le schéma de Leapfrog pour les équations de Shallow Water avec une grille décalée.
    Parameters:
    u : 2D array
        Zonal velocity component for all time steps at the interfaces grid points. (shape: (nt, nx))
    eta : 2D array
        elavation component for all time steps at the staggered grid points. (shape: (nt, nx))
    dt : float
        Time step size.
    dx : float
        Grid spacing in the x-direction.
    step : int
        Current time step index.
    """
    if step < 1:
        raise ValueError("Leapfrog scheme requires at least two previous time steps.")
    
    # Compléter ...

**Question 10 :** Comparer l'erreur de troncature spatiale (ordre 2 en $\Delta x$) de ce schéma à celle du précédent. Montrer que les deux schémas sont d'ordre 2 mais que leurs constantes d'erreur diffèrent. Quel pas de grille alternée $\Delta x_a$ permet d'obtenir la même précision que pour la grille non alternée de pas $\Delta x$ ?

**Question 11 :** Reproduire l’analyse de Von Neumann pour ce schéma et donner la condition CFL. En posant : 
$$
    u_{j+\frac12}^n = \hat u\, e^{i(k(j+\frac12)\Delta x - \omega n\Delta t)},
    \qquad
    \eta_j^n = \hat \eta\, e^{i(k j\Delta x - \omega n\Delta t)}.
$$


**Question 12 :**  La CFL est-elle plus restrictive sur la grille décalée si l'on choisit $\Delta x$ de sorte à ce que la précision des deux schémas d'ordre deux soient les mêmes ?

**Question 13 :** Comparer qualitativement les relations de dispersion des deux schémas et du cas analytique dans la limite $\omega \Delta t\rightarrow 0$. Discuter du comportement de la solution près de l’échelle de maille et des effets dispersifs à grande longueur d'onde.

**Question 14 :** Nous donnons la fonction RK2 pour initialiser note shéma numérique, une fonction permettant de créer un paquet d'onde gaussien : 

\begin{equation}
\exp\left[-(x - ct - \tfrac{L}{2})^2 / 2\sigma^2\right] \cos(k_0 (x - ct))
\end{equation}

 et un solveur en utilisant les fonctions précédemment définies. Visualisez les effet de dispersion associés aux méthodes présentées, en définissant correctement la variable $k_0$ (qui controle le nombre d'onde du paquet d'onde). 

- vous commenterez tout particulièrement la valeur $k_0$ = 0.5 et son comportement

In [17]:
def Runge_Kutta_init_staggered(u, eta, dt, dx):
    def rhs(u1, e1):
        deta_dx_u = (np.roll(e1, -1) - e1) / dx
        du_dx_e   = (u1 - np.roll(u1, 1)) / dx
        return (-G * deta_dx_u, -H * du_dx_e)

    k1u, k1e = rhs(u[0], eta[0])
    u_half   = u[0]   + 0.5*dt*k1u
    e_half   = eta[0] + 0.5*dt*k1e
    k2u, k2e = rhs(u_half, e_half)
    u[1]   = u[0]   + dt*k2u
    eta[1] = eta[0] + dt*k2e

In [ ]:
def wave_packet(x, t, k0, sigma): 
    c = np.sqrt(G * H)
    return np.exp(-((x - c*t) - L/2)**2 / (2*sigma**2)) * np.cos(k0 * (x - c*t))

In [ ]:
c = np.sqrt(G * H) 
k0_factor = 
sigma = 4

def run_packet(dx=0.2, dt=0.002, t_end=T):
    # grille décallée
    nx = int(L/dx)
    x_eta = np.arange(nx) * dx
    x_u   = (np.arange(nx) + 0.5) * dx

    # high-mode near Nyquist
    k0 = k0_factor * (np.pi/dx)
    eta0 = wave_packet(x_eta, 0, k0, sigma)
    u0 = (c/H) * eta0

# initialize wavepacket
    eta_bis = wave_packet(x_u, 0, k0, sigma)
    u0_stag   = (c/H)* eta_bis

    nt = int(t_end/dt) + 1
    t = np.arange(nt) * dt

    u  = np.zeros((nt, nx))
    eta  = np.zeros((nt, nx))
    
    u_stag = np.zeros((nt, nx))
    eta_stag = np.zeros((nt, nx))

    u[0] = u0
    eta[0] = eta0
    Runge_Kutta_init(u, eta, dt, dx)

    u_stag[0] = u0_stag
    eta_stag[0] = eta0
    Runge_Kutta_init_staggered(u_stag, eta_stag, dt, dx)

    for n in range(1, nt-1):
        leapfrog_step(u, eta, dt, dx, n)
        leapfrog_step_staggered(u_stag, eta_stag, dt, dx, n)

    return eta, eta_stag, u, u_stag, x_eta, t

dx = 0.01
dt = 0.0001 # satisfies both CFLs
eta, eta_stag, u, u_stag, x_eta, t = run_packet(dx=dx, dt=dt, t_end=T)

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(12, 5), dpi = 300)

x_mesh,t_mesh = np.meshgrid(x_eta, t)
eta_ana = wave_packet(x_mesh, t_mesh, k0_factor * (np.pi/dx), sigma)
snapshot_time_indices = [0, int(0.25*len(t)), int(0.5*len(t)), int(0.75*len(t)), -1]
cmap = plt.get_cmap("Greys", len(snapshot_time_indices))
colors = [cmap(i) for i in range(len(snapshot_time_indices))]

for i, time_index in enumerate(snapshot_time_indices):

    ax[0].plot(x_eta, eta_ana[time_index], lw=0.4, color=colors[i])
    ax[0].set_title("Solution Analytique")
    ax[1].plot(x_eta, eta_stag[time_index], lw=0.4, color=colors[i])
    ax[1].set_title("Solution Numérique (grille décalée)")
 
    ax[2].plot(x_eta, eta[time_index], lw=0.4, color=colors[i])
    ax[2].set_title("Solution Numérique (grille normale)")
fig.supxlabel("x")
fig.supylabel("$\\eta(x)$")
plt.tight_layout()
plt.show()

**Question 15 :** Conclure sur l’avantage pratique de la grille décalée pour les simulations.